# Trabalho 1: Do dado à decisão
**Ciência de Dados e IA**

Preencham os dados do grupo abaixo. Em todo o notebook:
- células de código com `____` devem ser completadas pelo grupo;
- células **Resposta** devem ser respondidas com as palavras do grupo, em até 5 linhas cada;
- não alterem os parâmetros da ficha do grupo.

| Campo | Preenchimento |
| --- | --- |
| Número do grupo | |
| Integrante 1 | |
| Integrante 2 | |
| Integrante 3 | |

## Configuração
Digitem o número do grupo (1 a 10) e executem as células desta seção.

In [ ]:
GRUPO = ____   # número do grupo, de 1 a 10

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings("ignore")

In [ ]:
# Parâmetros de todos os grupos. NÃO ALTERE.
PARAMS = {
 1: dict(dataset="penguins", alvo="species", evidencia="island", ev_valor="Biscoe", classe_alvo="Adelie",
         teste_var="body_mass_g", teste_grupo="sex", grupos=("Male","Female"), alfa=0.05,
         nova_priori=0.10, test_size=0.25, seed=11,
         features=["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]),
 2: dict(dataset="penguins", alvo="species", evidencia="island", ev_valor="Dream", classe_alvo="Adelie",
         teste_var="bill_depth_mm", teste_grupo="island", grupos=("Dream","Torgersen"), alfa=0.05,
         nova_priori=0.50, test_size=0.30, seed=22,
         features=["bill_length_mm","flipper_length_mm","body_mass_g"]),
 3: dict(dataset="mpg", alvo="origin", evidencia="cylinders", ev_valor=6, classe_alvo="usa",
         teste_var="mpg", teste_grupo="origin", grupos=("europe","japan"), alfa=0.01,
         nova_priori=0.30, test_size=0.20, seed=33,
         features=["mpg","displacement","horsepower","weight","acceleration"]),
 4: dict(dataset="mpg", alvo="origin", evidencia="cylinders", ev_valor=4, classe_alvo="japan",
         teste_var="acceleration", teste_grupo="origin", grupos=("usa","japan"), alfa=0.10,
         nova_priori=0.50, test_size=0.30, seed=44,
         features=["mpg","horsepower","weight","acceleration","model_year"]),
 5: dict(dataset="titanic", alvo="survived", evidencia="sex", ev_valor="female", classe_alvo=1,
         teste_var="age", teste_grupo="survived", grupos=(1,0), alfa=0.01,
         nova_priori=0.60, test_size=0.25, seed=55,
         features=["pclass","age","sibsp","parch","fare"]),
 6: dict(dataset="titanic", alvo="survived", evidencia="pclass", ev_valor=3, classe_alvo=1,
         teste_var="fare", teste_grupo="sex", grupos=("female","male"), alfa=0.01,
         nova_priori=0.20, test_size=0.20, seed=66,
         features=["pclass","age","fare","sibsp"]),
 7: dict(dataset="wine", alvo="cultivar", evidencia="alcohol_faixa", ev_valor="alto", classe_alvo="cultivar_0",
         teste_var="magnesium", teste_grupo="cultivar", grupos=("cultivar_0","cultivar_1"), alfa=0.05,
         nova_priori=0.10, test_size=0.25, seed=77,
         features=["alcohol","malic_acid","magnesium","color_intensity","proline"]),
 8: dict(dataset="wine", alvo="cultivar", evidencia="color_intensity_faixa", ev_valor="alto", classe_alvo="cultivar_2",
         teste_var="magnesium", teste_grupo="cultivar", grupos=("cultivar_1","cultivar_2"), alfa=0.05,
         nova_priori=0.60, test_size=0.30, seed=88,
         features=["alcohol","flavanoids","hue","color_intensity","proline","ash"]),
 9: dict(dataset="breast_cancer", alvo="diagnostico", evidencia="mean radius_faixa", ev_valor="alto", classe_alvo="maligno",
         teste_var="mean texture", teste_grupo="diagnostico", grupos=("maligno","benigno"), alfa=0.05,
         nova_priori=0.05, test_size=0.25, seed=99,
         features="todas"),
 10: dict(dataset="breast_cancer", alvo="diagnostico", evidencia="mean concave points_faixa", ev_valor="alto", classe_alvo="maligno",
         teste_var="mean fractal dimension", teste_grupo="diagnostico", grupos=("maligno","benigno"), alfa=0.01,
         nova_priori=0.50, test_size=0.20, seed=101,
         features=["mean radius","mean texture","mean perimeter","mean area","mean smoothness",
                   "mean compactness","mean concavity","mean concave points","mean symmetry","mean fractal dimension"]),
}

def carregar(grupo):
    p = PARAMS[grupo]
    d = p["dataset"]
    if d == "penguins":
        df = sns.load_dataset("penguins")
    elif d == "mpg":
        df = sns.load_dataset("mpg").drop(columns=["name"])
    elif d == "titanic":
        df = sns.load_dataset("titanic")[["survived","pclass","sex","age","sibsp","parch","fare","embarked"]]
    elif d == "wine":
        w = load_wine(as_frame=True); df = w.frame.rename(columns={"target":"cultivar"})
        df["cultivar"] = "cultivar_" + df["cultivar"].astype(str)
    else:
        b = load_breast_cancer(as_frame=True); df = b.frame.rename(columns={"target":"diagnostico"})
        df["diagnostico"] = df["diagnostico"].map({0:"maligno", 1:"benigno"})
    if p["evidencia"].endswith("_faixa"):
        col = p["evidencia"][:-6]
        df[p["evidencia"]] = np.where(df[col] > df[col].median(), "alto", "baixo")
    if p["features"] == "todas":
        p["features"] = [c for c in df.columns if c not in (p["alvo"], p["evidencia"])]
    return df, p

In [ ]:
dados, P = carregar(GRUPO)
print(f"FICHA DO GRUPO {GRUPO}")
print(f"Conjunto de dados ............ {P['dataset']}")
print(f"Variável alvo (y) ............ {P['alvo']}")
print(f"Etapa 2: teste de '{P['teste_var']}' entre {P['teste_grupo']} = {P['grupos'][0]} e {P['teste_grupo']} = {P['grupos'][1]}, com alfa = {P['alfa']}")
print(f"Etapa 3: calcular P({P['alvo']} = {P['classe_alvo']} | {P['evidencia']} = {P['ev_valor']}); nova priori = {P['nova_priori']}")
print(f"Etapa 4: test_size = {P['test_size']}, random_state = {P['seed']}")
print(f"Etapas 4 e 5: características usadas = {P['features']}")

## Etapa 1: Entender os dados
Antes de qualquer modelo, é preciso saber o que há na matriz de dados: quantas observações (n), quantas variáveis (p), qual é a resposta (y) e onde há problemas de qualidade.

In [ ]:
print("Dimensão:", dados.shape)
display(dados.head())
print(dados.dtypes)
print("\nValores ausentes por coluna:")
print(dados.isna().sum())

**1.1** Quanto valem n e p neste conjunto (sem contar a variável alvo)? O que é X e o que é y no problema do grupo?

**Resposta:** _escreva aqui_

**1.2** Classifiquem as variáveis em numéricas e categóricas. Os dados são estruturados ou não estruturados? Justifiquem.

**Resposta:** _escreva aqui_

In [ ]:
# Tratamento de valores ausentes: removemos as linhas com ausência nas colunas usadas no trabalho.
colunas_usadas = list(dict.fromkeys(P["features"] + [P["alvo"], P["evidencia"], P["teste_var"], P["teste_grupo"]]))
antes = len(dados)
dados = dados.dropna(subset=colunas_usadas).copy()
print(f"Linhas antes: {antes} | depois: {len(dados)} | removidas: {antes - len(dados)}")

**1.3** Quantas linhas foram removidas? Que efeito isso pode ter nas conclusões, e qual seria uma alternativa à remoção?

**Resposta:** _escreva aqui_

## Etapa 2: Inferência estatística
Vamos verificar se a média de uma variável difere entre dois grupos da população, usando o teste t de Welch (que não supõe variâncias iguais).

In [ ]:
grupo_A = dados.loc[dados[P["teste_grupo"]] == P["grupos"][0], P["teste_var"]]
grupo_B = dados.loc[dados[P["teste_grupo"]] == P["grupos"][1], P["teste_var"]]
print(f"Média em {P['grupos'][0]}: {grupo_A.mean():.3f} (n = {len(grupo_A)})")
print(f"Média em {P['grupos'][1]}: {grupo_B.mean():.3f} (n = {len(grupo_B)})")

# Complete com as duas amostras a comparar
resultado = stats.ttest_ind(____, ____, equal_var=False)
ic = resultado.confidence_interval(0.95)
print(f"Estatística t = {resultado.statistic:.3f} | valor-p = {resultado.pvalue:.6f}")
print(f"IC 95% da diferença entre as médias: [{ic.low:.3f}; {ic.high:.3f}]")

**2.1** Escrevam a hipótese nula (H0) e a hipótese alternativa (H1) do teste, usando os nomes das variáveis do grupo.

**Resposta:** _escreva aqui_

**2.2** Comparem o valor-p com o alfa da ficha. O grupo rejeita ou não rejeita H0? O que isso significa em linguagem simples?

**Resposta:** _escreva aqui_

**2.3** O que o intervalo de confiança de 95% indica? Ele contém o zero? Como isso se relaciona com a decisão do item 2.2?

**Resposta:** _escreva aqui_

**2.4** Neste problema, o que seria cometer um erro Tipo I? E um erro Tipo II?

**Resposta:** _escreva aqui_

## Etapa 3: Teorema de Bayes (cálculo à mão)
Esta etapa **não exige código novo**. Usem a tabela de contagem abaixo e façam as contas à mão, mostrando cada passo.

$$P(C \mid E) = \frac{P(E \mid C)\,P(C)}{P(E)}$$

Para a mudança de priori, usem:

$$P(C \mid E) = \frac{q \cdot P(E \mid C)}{q \cdot P(E \mid C) + (1-q) \cdot P(E \mid \text{não } C)}$$

onde $q$ é a nova priori da ficha.

In [ ]:
tabela = pd.crosstab(dados[P["evidencia"]], dados[P["alvo"]], margins=True, margins_name="Total")
print(f"Classe C = {P['alvo']} = {P['classe_alvo']} | Evidência E = {P['evidencia']} = {P['ev_valor']}")
display(tabela)

**3.1** A partir da tabela, calculem e mostrem a conta de: a priori P(C), a verossimilhança P(E | C), a evidência P(E) e a posterior P(C | E).

**Resposta:** _escreva aqui_

**3.2** Compare P(E | C) com P(C | E). Por que os dois valores são diferentes?

**Resposta:** _escreva aqui_

**3.3** Suponham que, em outra população, a priori de C fosse a nova priori da ficha (q). Calculem P(E | não C) pela tabela e a nova posterior. Expliquem por que ela mudou, se a evidência é a mesma.

**Resposta:** _escreva aqui_

## Etapa 4: Aprendizado supervisionado com Naive Bayes
Agora o computador faz o que fizemos à mão na Etapa 3, combinando várias características ao mesmo tempo.

In [ ]:
X = dados[P["features"]]
y = dados[P["alvo"]]

# Complete com os valores da ficha do grupo
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=____, random_state=____, stratify=y)

modelo = GaussianNB()
modelo.____(X_treino, y_treino)

acc_treino = accuracy_score(y_treino, modelo.predict(X_treino))
acc_teste = accuracy_score(y_teste, modelo.predict(X_teste))
print(f"Acurácia no treino: {acc_treino:.4f} | Acurácia no teste: {acc_teste:.4f}")
print("Matriz de confusão (teste), na ordem", list(modelo.classes_))
print(confusion_matrix(y_teste, modelo.predict(X_teste), labels=modelo.classes_))

**4.1** Que tipo de aprendizagem é esta e por quê? A tarefa é de detecção, reconhecimento ou previsão?

**Resposta:** _escreva aqui_

**4.2** Comparem as acurácias de treino e de teste. Há sinal de sobreajuste (overfitting)? Por que avaliamos o modelo em dados que ele não viu?

**Resposta:** _escreva aqui_

In [ ]:
corr = X.corr().abs()
corr = corr.where(~np.eye(len(corr), dtype=bool))   # ignora a diagonal
par = corr.stack().idxmax()
print(f"Par de características mais correlacionado: {par[0]} x {par[1]} (correlação = {X.corr().loc[par]:.3f})")

**4.3** O Naive Bayes supõe que as características são independentes dentro de cada classe. A correlação acima confirma ou contraria essa suposição? Mesmo assim, o modelo pode classificar bem? Justifiquem.

**Resposta:** _escreva aqui_

## Etapa 5: Aprendizado não supervisionado e espaço de características
Agora escondemos os rótulos. O K-means tenta formar grupos só com X. Depois comparamos esses grupos com as classes reais.
A **pureza** mede essa comparação: é a fração de observações que pertencem à classe majoritária do seu grupo (1,0 = grupos perfeitos).

In [ ]:
def pureza(clusters, classes):
    t = pd.crosstab(clusters, classes)
    return t.max(axis=1).sum() / t.values.sum()

k = y.nunique()   # número de grupos = número de classes

# Sem normalização
km_sem = KMeans(n_clusters=____, n_init=10, random_state=P["seed"]).fit(X)
print(f"Pureza SEM normalização: {pureza(km_sem.labels_, y):.4f}")

# Com normalização (média 0 e desvio padrão 1 em cada característica)
X_norm = StandardScaler().____(X)
km_com = KMeans(n_clusters=____, n_init=10, random_state=P["seed"]).fit(X_norm)
print(f"Pureza COM normalização: {pureza(km_com.labels_, y):.4f}")
display(pd.crosstab(km_com.labels_, y, rownames=["grupo K-means"], colnames=["classe real"]))

**5.1** Comparem a pureza com e sem normalização. Por que a escala das características muda o resultado do K-means? Citem duas características do grupo com escalas muito diferentes.

**Resposta:** _escreva aqui_

**5.2** Os grupos encontrados pelo K-means correspondem às classes reais? O que explica as diferenças?

**Resposta:** _escreva aqui_

In [ ]:
pca = PCA(n_components=____)
X_2d = pca.fit_transform(X_norm)
print(f"Variância explicada pelos 2 componentes: {pca.explained_variance_ratio_.sum():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=y, ax=ax[0]); ax[0].set_title("Classes reais")
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=km_com.labels_, palette="Set2", ax=ax[1]); ax[1].set_title("Grupos do K-means")
for a in ax: a.set_xlabel("Componente 1"); a.set_ylabel("Componente 2")
plt.tight_layout(); plt.show()

**5.3** O grupo partiu de p características e chegou a 2 componentes. Quanto da variância foi preservado? O que se ganha e o que se perde com essa redução?

**Resposta:** _escreva aqui_

## Reflexão final

**R1** Comparem a Etapa 2 com a Etapa 4: qual delas tem como objetivo entender uma relação nos dados, e qual tem como objetivo prever? Por que as duas são úteis?

**Resposta:** _escreva aqui_

**R2** Se o conjunto do grupo tivesse 10 vezes mais características e o mesmo número de observações, que problema poderia surgir? Como ele poderia ser reduzido?

**Resposta:** _escreva aqui_

**R3** Descrevam, em uma frase por etapa, o caminho do grupo: coleta, processamento, treinamento e aplicação. Em que situação real o modelo do grupo poderia ser usado?

**Resposta:** _escreva aqui_

## Declaração de contribuição
| Integrante | O que fez |
| --- | --- |
| | |
| | |
| | |